# Exploring the quickR API

A guided tour of the REST API (`/api/v1`) and the MCP server (`/mcp/`), run against a live
backend. Everything here happens on a throwaway topic that gets deleted at the end. The
seeded demo topics are only read.

**Start a backend first**, either of:

```bash
make up && make seed-docker          # full stack in docker, API on :8100
make db migrate api                  # or: local API with reload on :8100 (then `make seed`)
```

Then `make notebook`. Point it elsewhere with `QUICKR_API=http://host:port make notebook`.
The interactive docs are at `{QUICKR_API}/docs`.

In [ ]:
import json
import os
import uuid
from datetime import UTC, datetime, timedelta
from io import StringIO

import httpx
import pandas as pd

BASE = os.environ.get("QUICKR_API", "http://localhost:8100").rstrip("/")
api = httpx.Client(base_url=f"{BASE}/api/v1", timeout=10)


def show(r: httpx.Response, expect: int | None = None):
    """Print status + pretty JSON; fail loudly on an unexpected status."""
    print(r.request.method, r.request.url.path, "->", r.status_code)
    if expect is not None and r.status_code != expect:
        raise AssertionError(f"expected {expect}, got {r.status_code}: {r.text}")
    if r.headers.get("content-type", "").startswith("application/json"):
        body = r.json()
        print(json.dumps(body, indent=2)[:2500])
        return body
    return r.text


show(api.get("/health"), 200);

## Reference data

What question types exist (and their options), and the server's size limits.

In [ ]:
types = show(api.get("/question-types"), 200)

In [ ]:
show(api.get("/limits"), 200);

## Topics

`GET /topics` sorts by `due` by default: due topics first, then scheduled topics by
`next_due_at`, then the rest by latest activity.

In [ ]:
topics = api.get("/topics").raise_for_status().json()
pd.DataFrame(
    [
        {
            "slug": t["slug"],
            "name": t["name"],
            "version": t["current_version"],
            "is_due": (t.get("due") or {}).get("is_due"),
            "next_due_at": (t.get("due") or {}).get("next_due_at"),
            "schedule": (t.get("schedule") or {}).get("kind"),
        }
        for t in topics["items"]
    ]
)

### Create a scratch topic

Each question has a `key` (stable across versions), a `label` and a `type`. Here: a
double with a unit, a bounded int, a single-select enum, a multi-select enum, and an
optional string. The slug gets a random suffix so reruns don't collide.

In [ ]:
SLUG = f"notebook-{uuid.uuid4().hex[:6]}"
QUESTIONS = [
    {"key": "coffee_ml", "label": "Coffee", "type": "double", "min": 0, "max": 2000, "unit": "ml"},
    {"key": "focus", "label": "Focus (1-5)", "type": "int", "min": 1, "max": 5},
    {
        "key": "place",
        "label": "Where",
        "type": "enum",
        "options": [{"value": "home", "label": "Home"}, {"value": "office", "label": "Office"}],
    },
    {
        "key": "with",
        "label": "Had it with",
        "type": "enum",
        "multiple": True,
        "required": False,
        "options": [
            {"value": "milk", "label": "Milk"},
            {"value": "sugar", "label": "Sugar"},
            {"value": "cake", "label": "Cake"},
        ],
    },
    {"key": "note", "label": "Note", "type": "str", "required": False},
]

topic = show(
    api.post("/topics", json={"name": "Notebook coffee", "slug": SLUG, "questions": QUESTIONS}),
    201,
)

## Entries

`answers` maps question key to value. A multi-select takes a list. Sending your own `id`
makes the request idempotent: resending it returns the stored entry instead of a
duplicate. That is how the phone's offline outbox retries safely. `captured_at` is the
client's clock and can be in the past.

In [ ]:
entry_id = str(uuid.uuid4())
body = {
    "id": entry_id,
    "answers": {"coffee_ml": 250, "focus": 4, "place": "office", "with": ["milk"]},
    "source": "notebook",
    "notes": "first cup",
}
first = show(api.post(f"/topics/{SLUG}/entries", json=body), 201)
again = api.post(f"/topics/{SLUG}/entries", json=body).raise_for_status().json()
print("\nresend returned the same entry:", again["id"] == first["id"] == entry_id)

In [ ]:
# A week of backdated entries, so stats and series have something to show.
import random

rng = random.Random(7)
now = datetime.now(UTC)
for day in range(1, 8):
    for hour in (9, 14):
        api.post(
            f"/topics/{SLUG}/entries",
            json={
                "answers": {
                    "coffee_ml": rng.choice([150, 200, 250, 350]),
                    "focus": rng.randint(2, 5),
                    "place": rng.choice(["home", "office"]),
                    "with": rng.sample(["milk", "sugar", "cake"], k=rng.randint(0, 2)),
                },
                "captured_at": (now - timedelta(days=day)).replace(hour=hour, minute=0).isoformat(),
                "source": "notebook",
            },
        ).raise_for_status()

entries = api.get(f"/topics/{SLUG}/entries", params={"limit": 50}).raise_for_status().json()
print(f"{entries['total']} entries; answers come back as a list of {{key, type, value}}")
pd.DataFrame(
    [
        {"captured_at": e["captured_at"], "channel": e["channel"], "version": e["version"]}
        | {a["key"]: a["value"] for a in e["answers"]}
        for e in entries["items"]
    ]
).head(10)

### Validation errors

Errors come back in one problem envelope, with an `errors` list of `{loc, message}` that
points at the offending field.

In [ ]:
show(
    api.post(f"/topics/{SLUG}/entries", json={"answers": {"coffee_ml": -5, "focus": 9, "place": "moon"}}), 422
);

### Edit and delete

`PATCH /entries/{id}` merges `answers` into the existing ones (`null` clears an answer).

In [ ]:
show(api.patch(f"/entries/{entry_id}", json={"answers": {"focus": 5, "with": None}, "notes": "edited"}), 200);

In [ ]:
throwaway = api.post(
    f"/topics/{SLUG}/entries", json={"answers": {"coffee_ml": 1, "focus": 1, "place": "home"}}
).json()
show(api.delete(f"/entries/{throwaway['id']}"), 204)
show(api.get(f"/entries/{throwaway['id']}"), 404);

## Stats, series and CSV

Stats summarise each question (numeric: count/min/max/mean, enum: per-option counts). A
series buckets one question over time (`bucket`=hour/day/week/month, in an IANA `tz`).

In [ ]:
stats = show(api.get(f"/topics/{SLUG}/stats"), 200)

In [ ]:
series = (
    api.get(f"/topics/{SLUG}/questions/coffee_ml/series", params={"bucket": "day", "tz": "Europe/London"})
    .raise_for_status()
    .json()
)
print(json.dumps(series, indent=2)[:800])

In [ ]:
csv = api.get(f"/topics/{SLUG}/entries.csv").raise_for_status().text
pd.read_csv(StringIO(csv)).head()

## Versions

Questions change by publishing a new version. `versions/check` is a dry run that reports
what would change and any warnings. Every entry stays pinned to the version it was
recorded against.

In [ ]:
v2 = [q for q in QUESTIONS if q["key"] != "note"] + [
    {
        "key": "decaf",
        "label": "Decaf?",
        "type": "enum",
        "required": False,
        "options": [{"value": "yes", "label": "Yes"}, {"value": "no", "label": "No"}],
    },
]
show(api.post(f"/topics/{SLUG}/versions/check", json={"questions": v2}), 200);

In [ ]:
show(
    api.post(
        f"/topics/{SLUG}/versions",
        json={"questions": v2, "base_version": 1, "change_note": "notebook: swap note for decaf"},
    ),
    201,
)
[v["version"] for v in api.get(f"/topics/{SLUG}/versions").json()]

## Schedules and due status

A schedule makes a topic "due". A new or changed schedule is never due straight away;
`next_due_at` shows when it will be.

In [ ]:
schedule = {"kind": "daily", "times": ["09:00", "15:00"], "timezone": "Europe/London", "notify": False}
t = api.patch(f"/topics/{SLUG}", json={"schedule": schedule}).raise_for_status().json()
print(json.dumps({"schedule": t["schedule"], "due": t["due"]}, indent=2))

In [ ]:
# Nothing from this topic yet: the schedule was just set.
due = api.get("/topics", params={"due_only": True}).raise_for_status().json()
[t["slug"] for t in due["items"]]

## Push

The VAPID public key a browser needs for `PushManager.subscribe`. Subscribing needs a
real browser, so this only reads the config.

In [ ]:
show(api.get("/push/config"));

## MCP

The same services are exposed as MCP tools over streamable HTTP at `/mcp/`, which is what
an agent (Claude, etc.) talks to. Entries recorded here get `channel: mcp`.

In [ ]:
from fastmcp import Client

async with Client(f"{BASE}/mcp/") as mcp:
    tools = await mcp.list_tools()
    print([t.name for t in tools])
    rec = await mcp.call_tool(
        "record_entry", {"ref": SLUG, "answers": {"coffee_ml": 90, "focus": 3, "place": "home"}}
    )
    print("\nrecorded via MCP, channel =", rec.structured_content["channel"])
    due_now = await mcp.call_tool("list_due_topics", {})
    print("due via MCP:", due_now.structured_content)

## Clean up

Only archived topics can be deleted.

In [ ]:
show(api.post(f"/topics/{SLUG}/archive"), 200)
show(api.delete(f"/topics/{SLUG}"), 204)
show(api.get(f"/topics/{SLUG}"), 404);